# TRELLIS.2 GGUF — Colab T4

**Среда выполнения → Сменить среду выполнения → T4 GPU. Две основные ячейки: «Установка» и «Запуск / перезапуск».**

Установку выполните один раз на новую среду. Она сама последовательно пройдёт три этапа: окружение, зависимости, готовые wheels/сборки. Первичная сборка может занять десятки минут. Кэш Google Drive включён по умолчанию и потребует подключения Диска: уже собранные wheels сохраняются в MyDrive/Trellis2-T4-cache/wheels и переиспользуются в следующей среде с теми же исходниками и ABI. Отключите USE_DRIVE_CACHE, если не хотите подключать Drive; тогда кэш остаётся только в текущей среде.

Для каждого запуска или перезапуска сервера используйте только вторую ячейку — она не устанавливает пакеты и не повторяет сборки. **Проверки CUDA и прогрев удаления фона вынесены в отдельные необязательные ячейки, по умолчанию выключенные.** Ключи GitHub/Hugging Face не нужны для предусмотренных публичных источников.

Основано на вашем Trellis2_GGUF_Colab(11).ipynb. Исправления: фиксированные Python/Torch/CUDA, обязательный GGUF-загрузчик, сборки T4 с постоянным кэшем, необязательные CUDA smoke tests, отменяемое удаление фона, HTTP/2 и запуск без занятой ячейки. Исходные Git-ревизии закреплены в colab/sources.lock.json. Зависимости Python разрешаются с жёсткими ограничениями Torch; диагностическая ячейка сохраняет полный список в /content/t4-environment.txt.

В интерфейсе откройте **Workflows → Trellis2-T4 → Only_Mesh_Simple** и загрузите свою картинку. Для PNG с прозрачностью повторное удаление фона пропускается. Модели Trellis/DINO скачиваются при первом запуске workflow; это не время загрузки самого интерфейса.

Примечание: реальные генерация/GLB-экспорт на T4 ещё должны быть проверены. При ошибках используйте отдельную диагностическую ячейку. Qwen-вариант требует дополнительных сторонних нод/весов и не является базовым профилем T4. Дополнительный nvdiffrec PBR-preview в эту установку не включён.


## 1. Установка — один раз на новую среду

In [ ]:
# УСТАНОВКА. Выберите T4 в настройках среды и нажмите ▶ здесь.
# Первый запуск собирает CUDA-модули. Повторный использует сохранённые этапы.
USE_DRIVE_CACHE = True # @param {type:"boolean"}

import base64, hashlib, io, json, os, pathlib, subprocess, sys, time, urllib.request, zipfile
ROOT = pathlib.Path('/content')
if not ROOT.exists():
    raise RuntimeError('Этот notebook предназначен для Google Colab с T4 GPU.')
APP = ROOT / 'trellis-t4-bootstrap'
APP.mkdir(exist_ok=True)
PAYLOAD = '...'
bundle = base64.b64decode(PAYLOAD)
fingerprint = hashlib.sha256(bundle).hexdigest()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for item in archive.infolist():
        if not (APP/item.filename).resolve().is_relative_to(APP.resolve()):
            raise ValueError('Некорректный путь в пакете')
    archive.extractall(APP)

def command(args, timeout=None):
    # Colab needs a PIPE + print: inherited file descriptors can hide subprocess logs.
    sys.path.insert(0,str(APP/'colab'))
    from process_output import run_streamed
    name=str(args[-1])
    return run_streamed(args,ROOT/f'colab-install-{name}.log',timeout=timeout)

def main():
    state = ROOT/'trellis-t4-processes.json'
    if state.exists():
        raise RuntimeError('Сначала остановите ComfyUI ячейкой остановки. Для обычного перезапуска используйте ячейку запуска, а не установку.')
    if USE_DRIVE_CACHE:
        from google.colab import drive
        drive.mount('/content/drive')
        cache=pathlib.Path('/content/drive/MyDrive/Trellis2-T4-cache/wheels')
        cache.mkdir(parents=True,exist_ok=True)
        os.environ['TRELLIS_WHEEL_CACHE']=str(cache)
        print('Постоянный кэш wheels:',cache,flush=True)
    else:
        os.environ.pop('TRELLIS_WHEEL_CACHE',None)
    marker_dir=ROOT/'t4-stage-cache';marker_dir.mkdir(exist_ok=True)
    lock=json.loads((APP/'colab/sources.lock.json').read_text())
    for manifest_name in ['t4-installed.json','t4-verified.json']:
        manifest=ROOT/manifest_name
        if manifest.exists() and pathlib.Path('/content/venv312/bin/python').exists():
            try:already_installed=json.loads(manifest.read_text()).get('sources')==lock
            except ValueError:already_installed=False
            if already_installed:
                print('Это окружение уже установлено с теми же версиями. Этапы установки пропущены.',flush=True)
                if USE_DRIVE_CACHE:
                    print('Сохраняю/сверяю готовые wheels в Drive; совместимые пакеты не пересобираются.',flush=True)
                    command([sys.executable,APP/'colab/setup_t4.py','builds'])
                print('Готово. Используйте отдельную ячейку запуска ComfyUI.',flush=True)
                return
    stage_inputs={
        'system':{'schema':2,'python':'3.12','torch':'2.9.1+cu126','cuda':'12.6','eigen':True},
        'dependencies':{'schema':2,**{k:lock[k] for k in ['comfy','trellis','gguf','utils3d']}},
        'builds':{'schema':2,'arch':'7.5','torch':'2.9.1+cu126',**{k:lock[k] for k in ['flex','cumesh','voxel','diffrast']}},
    }
    stages=[('system','1/3 — Python 3.12, Torch cu126, CUDA-компилятор 12.6'),
            ('dependencies','2/3 — Репозитории, ComfyUI-GGUF, зависимости и профили'),
            ('builds','3/3 — Готовые wheels из кэша или сборка отсутствующих')]
    for name,title in stages:
        stage_fingerprint=hashlib.sha256(json.dumps(stage_inputs[name],sort_keys=True).encode()).hexdigest()
        marker=marker_dir/(name+'.json')
        complete=False
        if marker.exists():
            try:complete=json.loads(marker.read_text()).get('fingerprint')==stage_fingerprint
            except ValueError:pass
        print('\n'+title,flush=True)
        if complete and pathlib.Path('/content/venv312/bin/python').exists() and not (name=='builds' and USE_DRIVE_CACHE):
            print('Этап уже завершён, использую кэш.',flush=True)
        else:
            # Invalidates the verification flag before changing any packages.
            (ROOT/'t4-verified.json').unlink(missing_ok=True)
            (ROOT/'t4-installed.json').unlink(missing_ok=True)
            command([sys.executable,APP/'colab/setup_t4.py',name])
            marker.write_text(json.dumps({'fingerprint':stage_fingerprint}))
    (ROOT/'t4-installed.json').write_text(json.dumps({'sources':lock,'torch':'2.9.1+cu126','gpu_arch':'7.5'},indent=2))
    print('\nУстановка готова. Теперь выполните отдельную ячейку «Запуск / перезапуск ComfyUI».')

try:
    main()
except KeyboardInterrupt:
    # Stop only the processes owned by this notebook, if they were started.
    subprocess.run([sys.executable,str(APP/'colab/server_t4.py'),'stop'])
    print('Остановлено. Завершённые этапы сохранены; нажмите ▶ для продолжения.')
    raise
except Exception:
    print('Этап завершился ошибкой. Причина указана выше. После исправления нажмите ▶ снова; завершённые этапы будут пропущены.')
    raise


## 2. Запуск / перезапуск ComfyUI
Запускает только сервер и новый HTTP/2-туннель. Если сервер этого notebook уже работает, он будет перезапущен. Ячейка завершится после появления ссылки; ComfyUI продолжит работать и не займёт очередь выполнения ячеек. При удалении всей среды Colab сначала снова потребуется установка.


In [ ]:
from pathlib import Path
import sys, re, importlib.util
from IPython.display import display, HTML
bootstrap=Path('/content/trellis-t4-bootstrap/colab')
if not Path('/content/venv312/bin/python').exists() or not Path('/content/ComfyUI/main.py').exists():
    raise RuntimeError('Сначала выполните установку в этой среде Colab.')
bootstrap.mkdir(parents=True,exist_ok=True)
launcher=bootstrap/'server_t4.py'
helper=bootstrap/'process_output.py'
launcher.write_text('"""Start returns to the notebook; stop targets only recorded matching processes."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.request\n\nROOT=Path(\'/content\')\nSTATE=ROOT/\'trellis-t4-processes.json\'\nCOMFY=ROOT/\'ComfyUI\'\nPY=ROOT/\'venv312/bin/python\'\nLOG=ROOT/\'comfy-t4.log\'\nTLOG=ROOT/\'cloudflare-t4.log\'\n\ndef identity(pid):\n    try:\n        return {\'cmd\':Path(f\'/proc/{pid}/cmdline\').read_bytes().decode().replace(\'\\x00\',\' \'),\n                \'started\':Path(f\'/proc/{pid}/stat\').read_text().split()[21]}\n    except (FileNotFoundError,ProcessLookupError):return None\n\ndef stop():\n    state=json.loads(STATE.read_text()) if STATE.exists() else {}\n    for kind in (\'tunnel\',\'comfy\'):\n        record=state.get(kind)\n        if not record:continue\n        pid=record[\'pid\']\n        if identity(pid)!=record[\'identity\']:\n            print(kind,\': recorded process already stopped or changed\');continue\n        os.killpg(pid,signal.SIGTERM)\n        for _ in range(30):\n            if identity(pid)!=record[\'identity\']:break\n            time.sleep(.1)\n        if identity(pid)==record[\'identity\']:os.killpg(pid,signal.SIGKILL)\n        print(\'Stopped\',kind)\n    STATE.unlink(missing_ok=True)\n\ndef spawn(args,log,cwd,env):\n    with log.open(\'w\') as output:\n        proc=subprocess.Popen(args,cwd=cwd,env=env,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)\n    return proc,{\'pid\':proc.pid,\'identity\':identity(proc.pid)}\n\ndef start():\n    if not (ROOT/\'t4-installed.json\').exists() and not (ROOT/\'t4-verified.json\').exists():\n        raise RuntimeError(\'Complete installation before launching. CUDA smoke tests are optional.\')\n    stop()\n    with socket.socket() as sock:\n        if sock.connect_ex((\'127.0.0.1\',8188))==0:raise RuntimeError(\'Port 8188 is occupied by a different server; stop its original launch cell first.\')\n    binary=ROOT/\'cloudflared-t4\'\n    if not binary.exists():\n        url=\'https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64\'\n        tmp=binary.with_suffix(\'.download\')\n        urllib.request.urlretrieve(url,tmp);tmp.chmod(0o755);tmp.replace(binary)\n    env=dict(os.environ,CUDA_HOME=\'/usr/local/cuda-12.6\',TORCH_CUDA_ARCH_LIST=\'7.5\',\n             SPARSE_CONV_BACKEND=\'flex_gemm\',SPARSE_ATTN_BACKEND=\'sdpa\',ATTN_BACKEND=\'sdpa\',\n             OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n    env[\'PATH\']=env[\'CUDA_HOME\']+\'/bin:\'+env.get(\'PATH\',\'\')\n    state={}\n    try:\n        # Obtain the public URL first; wildcard CORS is the user\'s requested launch mode.\n        tunnel,record=spawn([str(binary),\'tunnel\',\'--protocol\',\'http2\',\'--url\',\'http://127.0.0.1:8188\'],TLOG,ROOT,env)\n        state[\'tunnel\']=record;STATE.write_text(json.dumps(state))\n        public_url=None\n        for _ in range(60):\n            if tunnel.poll() is not None:raise RuntimeError(TLOG.read_text(errors=\'replace\')[-4000:])\n            match=re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\',TLOG.read_text(errors=\'replace\'))\n            if match:\n                public_url=match.group(0)\n                break\n            time.sleep(1)\n        if public_url is None:raise TimeoutError(\'No tunnel URL; see cloudflare-t4.log\')\n        comfy,record=spawn([str(PY),\'-u\',\'main.py\',\'--listen\',\'127.0.0.1\',\'--port\',\'8188\',\n                            \'--disable-pinned-memory\',\'--enable-cors-header\',\'*\'],LOG,COMFY,env)\n        state[\'comfy\']=record;STATE.write_text(json.dumps(state))\n        for _ in range(150):\n            if comfy.poll() is not None:raise RuntimeError(\'ComfyUI exited: \'+LOG.read_text(errors=\'replace\')[-8000:])\n            try:\n                with urllib.request.urlopen(\'http://127.0.0.1:8188/object_info\',timeout=2) as response:info=json.load(response)\n                required=[\'Trellis2LoadModel_GGUF\',\'Trellis2PreProcessImage_GGUF\',\'Trellis2MeshWithVoxelGenerator_GGUF\']\n                if any(k not in info for k in required):raise RuntimeError(\'Trellis nodes failed to import. See /content/comfy-t4.log\')\n                if \'Using native ComfyUI-GGUF support\' not in LOG.read_text(errors=\'replace\'):\n                    raise RuntimeError(\'Native GGUF loader not active. See /content/comfy-t4.log\')\n                break\n            except (TimeoutError,OSError):time.sleep(1)\n        else:raise TimeoutError(\'ComfyUI did not become ready in time; see log.\')\n        request=urllib.request.Request(\'http://127.0.0.1:8188/\',headers={\'Sec-Fetch-Site\':\'cross-site\'})\n        with urllib.request.urlopen(request,timeout=5) as response:\n            if response.status!=200:raise RuntimeError(\'Browser navigation check failed\')\n        print(\'\\nComfyUI:\',public_url,flush=True)\n        print(\'Ready. This cell finishes; ComfyUI stays running. Use the stop cell before setup changes.\')\n        print(\'Templates: Workflows > Trellis2-T4. Start with Only_Mesh_Simple.\')\n    except BaseException:\n        stop();raise\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'action\',choices=[\'start\',\'stop\'])\n    action=parser.parse_args().action\n    globals()[action]()\n',encoding='utf8')
helper.write_text('"""Stream subprocess output into notebook stdout and a persistent text log."""\nimport os\nfrom pathlib import Path\nimport queue\nimport signal\nimport subprocess\nimport threading\nimport time\n\n\ndef run_streamed(args, log_path, timeout=None, env=None):\n    args = list(map(str, args))\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    environment = dict(os.environ if env is None else env, PYTHONUNBUFFERED=\'1\')\n    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               text=True, encoding=\'utf8\', errors=\'replace\', bufsize=1,\n                               env=environment, start_new_session=os.name != \'nt\')\n    output = queue.Queue()\n    def read_output():\n        try:\n            for line in process.stdout:\n                output.put(line)\n        finally:\n            output.put(None)\n    reader = threading.Thread(target=read_output, daemon=True)\n    reader.start()\n    start = time.monotonic()\n    try:\n        with log_path.open(\'a\', encoding=\'utf8\', buffering=1) as log:\n            log.write(\'\\nCOMMAND: \'+\' \'.join(args)+\'\\n\')\n            ended = False\n            while not ended:\n                if timeout is not None and time.monotonic()-start > timeout:\n                    raise subprocess.TimeoutExpired(args, timeout)\n                try:\n                    line = output.get(timeout=.2)\n                except queue.Empty:\n                    continue\n                if line is None:\n                    ended = True\n                else:\n                    print(line, end=\'\', flush=True)\n                    log.write(line)\n            code = process.wait(timeout=5)\n            log.write(f\'EXIT CODE: {code}\\n\')\n        if code:\n            print(f\'Full subprocess log: {log_path}\', flush=True)\n            raise subprocess.CalledProcessError(code, args)\n        return subprocess.CompletedProcess(args, code)\n    finally:\n        if process.poll() is None:\n            if os.name == \'nt\': process.terminate()\n            else: os.killpg(process.pid, signal.SIGTERM)\n            try:\n                process.wait(timeout=3)\n            except subprocess.TimeoutExpired:\n                if os.name == \'nt\': process.kill()\n                else: os.killpg(process.pid, signal.SIGKILL)\n                process.wait()\n        reader.join(timeout=2)\n        process.stdout.close()\n',encoding='utf8')
spec=importlib.util.spec_from_file_location('trellis_launcher_output',helper)
output=importlib.util.module_from_spec(spec)
spec.loader.exec_module(output)
result=output.run_streamed([sys.executable,'-u',str(launcher),'start'],'/content/colab-launch.log')
urls=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',Path('/content/cloudflare-t4.log').read_text())
if not urls:
    raise RuntimeError('Туннель не выдал ссылку. Посмотрите /content/cloudflare-t4.log')
display(HTML(f'<p><a href="{urls[-1]}" target="_blank" rel="noopener" style="display:inline-block;padding:16px 28px;border-radius:10px;background:#2563eb;color:white;font-size:22px;text-decoration:none">Открыть ComfyUI ↗</a></p><p>{urls[-1]}</p>'))


## Необязательно: диагностика CUDA
Только при ошибках или для проверки новой сборки. Не выполняется автоматически. Отметьте RUN_DIAGNOSTICS и запустите эту ячейку.


In [ ]:
RUN_DIAGNOSTICS = False # @param {type:"boolean"}
if RUN_DIAGNOSTICS:
    import sys
    from pathlib import Path
    sys.path.insert(0,'/content/trellis-t4-bootstrap/colab')
    from process_output import run_streamed
    result=run_streamed([sys.executable,'-u','/content/trellis-t4-bootstrap/colab/setup_t4.py','verify'],'/content/colab-diagnostics.log')


## Необязательно: заранее загрузить модель удаления фона
Это бывший этап 5/5. Нужен только для предварительного скачивания u2net. Если пропустить, модель скачается при первом удалении фона. Для PNG с прозрачностью эта модель не вызывается.


In [ ]:
PRELOAD_BACKGROUND = False # @param {type:"boolean"}
if PRELOAD_BACKGROUND:
    import os, sys
    sys.path.insert(0,'/content/trellis-t4-bootstrap/colab')
    from process_output import run_streamed
    result=run_streamed(['/content/venv312/bin/python','-u','-c',"from rembg import new_session; new_session('u2net',providers=['CPUExecutionProvider']); print('Модель удаления фона готова.')"],'/content/colab-background.log',env=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2'),timeout=180)


## Необязательно: остановить только ComfyUI и туннель
Отметьте STOP_COMFY и выполните эту ячейку. Установленные пакеты и модели сохранятся до удаления среды Colab. После остановки используйте ячейку 2 для повторного запуска. По умолчанию эта ячейка ничего не останавливает при «Выполнить все».


In [ ]:
STOP_COMFY = False # @param {type:"boolean"}
if STOP_COMFY:
    import subprocess, sys
    subprocess.run([sys.executable, '/content/trellis-t4-bootstrap/colab/server_t4.py', 'stop'], check=True)


## Необязательно: последние строки логов

In [ ]:
from pathlib import Path
for name in ['comfy-t4.log','cloudflare-t4.log']:
    p=Path('/content')/name
    print('\n'+name+'\n'+(p.read_text(errors='replace')[-12000:] if p.exists() else 'Лог ещё не создан'))


## Необязательно: скачать результаты
Перед завершением Colab сохраните GLB на компьютер. /content не является постоянным хранилищем. ZIP создаётся вне папки output.


In [ ]:
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    from pathlib import Path
    import shutil
    from google.colab import files
    out=Path('/content/ComfyUI/output')
    if not out.exists():
        raise RuntimeError('Папка результатов ещё не создана')
    archive=shutil.make_archive('/content/trellis-results','zip',out)
    files.download(archive)


## Необязательно: сохранить уже собранные wheels в Drive
Для текущей среды, где 40-минутная сборка уже завершена. Отметьте SAVE_EXISTING_WHEELS и выполните ячейку: она только сохранит готовые файлы и не запустит установку. В новой среде оставьте USE_DRIVE_CACHE включённым, чтобы восстановить их.


In [ ]:
SAVE_EXISTING_WHEELS = False # @param {type:"boolean"}
if SAVE_EXISTING_WHEELS:
    import json, importlib.util
    from pathlib import Path
    from google.colab import drive
    drive.mount('/content/drive')
    helper=Path('/content/trellis-wheel-cache.py')
    helper.write_text('"""Optional persistent cache of compiled wheels, keyed by source and runtime ABI."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport uuid\n\n\ndef sha256(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass WheelCache:\n    def __init__(self, root=None):\n        self.root=Path(root) if root else None\n\n    def directory(self, package, identity):\n        if not self.root:return None\n        runtime={\'machine\':platform.machine(),\'libc\':platform.libc_ver()}\n        key=hashlib.sha256(json.dumps({\'identity\':identity,\'runtime\':runtime},sort_keys=True).encode()).hexdigest()[:24]\n        return self.root/package/key\n\n    def restore(self, package, identity, destination):\n        directory=self.directory(package,identity)\n        if directory is None or not (directory/\'manifest.json\').exists():return None\n        try:\n            info=json.loads((directory/\'manifest.json\').read_text())\n        except (OSError,ValueError):\n            print(\'Unreadable persistent cache manifest:\',directory,flush=True);return None\n        if info.get(\'identity\')!=identity:return None\n        name=info.get(\'wheel\',\'\')\n        if Path(name).name!=name or not name.endswith(\'.whl\'):raise ValueError(\'Invalid wheel cache filename\')\n        source=directory/name\n        if not source.exists() or sha256(source)!=info[\'sha256\']:\n            print(\'Persistent wheel cache is incomplete/corrupt:\',source,flush=True);return None\n        destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n        target=destination/name\n        shutil.copy2(source,target)\n        print(\'Restored compiled wheel:\',target.name,flush=True)\n        return target\n\n    def save(self, package, identity, wheel):\n        directory=self.directory(package,identity)\n        if directory is None:return\n        directory.mkdir(parents=True,exist_ok=True)\n        wheel=Path(wheel);target=directory/wheel.name\n        digest=sha256(wheel)\n        if not target.exists() or sha256(target)!=digest:\n            temporary=directory/(wheel.name+\'.\'+uuid.uuid4().hex+\'.tmp\')\n            shutil.copy2(wheel,temporary);temporary.replace(target)\n        manifest={\'identity\':identity,\'wheel\':wheel.name,\'sha256\':digest}\n        temp=directory/(\'manifest.\'+uuid.uuid4().hex+\'.tmp\')\n        temp.write_text(json.dumps(manifest,indent=2));temp.replace(directory/\'manifest.json\')\n        print(\'Persistent wheel saved:\',target,flush=True)\n',encoding='utf8')
    spec=importlib.util.spec_from_file_location('trellis_wheel_cache_export',helper)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    cache=module.WheelCache('/content/drive/MyDrive/Trellis2-T4-cache/wheels')
    saved=0
    for marker in Path('/content/t4-wheelhouse').glob('*-build.json'):
        info=json.loads(marker.read_text())
        wheel=Path(info['wheel'])
        if wheel.exists():
            cache.save(marker.name.removesuffix('-build.json'),info['identity'],wheel)
            saved+=1
    print(f'Сохранено готовых wheels: {saved}. Установка и сборка не запускались.')


## Необязательно: отчёт о системе и текущем выполнении
Отметьте COLLECT_REPORT и выполните ячейку во время задержки Generator. Отчёт читает версии, настройки, память и nvidia-smi; не запускает CUDA-кернелы, не устанавливает пакеты и не перезапускает ComfyUI. Файл trellis-diagnostics.json скачается на компьютер, его можно приложить к сообщению.


In [ ]:
COLLECT_REPORT = False # @param {type:"boolean"}
if COLLECT_REPORT:
    import sys, subprocess
    from pathlib import Path
    from google.colab import files
    script=Path('/content/trellis-collect-diagnostics.py')
    script.write_text('"""Read-only report: no installs, no CUDA kernels, no model loads, no restarts."""\nimport argparse\nimport ast\nimport importlib.metadata as metadata\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport time\nimport urllib.request\n\nROOT=Path(\'/content\')\nNODE=ROOT/\'ComfyUI/custom_nodes/ComfyUI-Trellis2-GGUF\'\n\ndef command(args,timeout=6):\n    try:\n        p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout)\n        return {\'code\':p.returncode,\'output\':p.stdout[-16000:]}\n    except (OSError,subprocess.TimeoutExpired) as error:return {\'error\':str(error)}\n\ndef redact(text):\n    return re.sub(r\'\\b(?:ghp_|github_pat_|hf_)[A-Za-z0-9_]+\',\'[REDACTED]\',text)\n\ndef text(path,limit=12000):\n    try:return redact(Path(path).read_text(errors=\'replace\')[-limit:])\n    except OSError:return None\n\ndef collect(samples=10):\n    report={\'time_utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\'note\':\'No CUDA inference or installations were performed.\'}\n    report[\'gpu\']=command([\'nvidia-smi\'])\n    report[\'nvcc\']=command([\'/usr/local/cuda-12.6/bin/nvcc\',\'--version\'])\n    report[\'memory\']=text(\'/proc/meminfo\',4000)\n    report[\'cpu\']={\'logical_count\':os.cpu_count(),\'quota\':text(\'/sys/fs/cgroup/cpu.max\'),\n                   \'cpuset\':text(\'/sys/fs/cgroup/cpuset.cpus.effective\')}\n    cpuinfo=text(\'/proc/cpuinfo\',100000) or \'\'\n    report[\'cpu\'][\'model\']=next((line.split(\':\',1)[1].strip() for line in cpuinfo.splitlines()\n                               if line.startswith(\'model name\')),None)\n    process_pid=None\n    try:\n        state=json.loads((ROOT/\'trellis-t4-processes.json\').read_text())\n        process_pid=int(state[\'comfy\'][\'pid\'])\n        report[\'comfy_pid\']=process_pid\n    except (OSError,ValueError,KeyError):pass\n    report[\'cgroup\']={name:text(\'/sys/fs/cgroup/\'+name) for name in [\'memory.events\',\'memory.current\',\'memory.max\']}\n    report[\'packages\']={}\n    for name in [\'torch\',\'torchvision\',\'torchaudio\',\'triton\',\'flex_gemm\',\'cumesh\',\'o_voxel\',\'nvdiffrast\',\'gguf\',\'rembg\',\'onnxruntime\',\'onnxruntime-gpu\',\'comfyui-frontend-package\']:\n        try:\n            package=metadata.distribution(name)\n            direct=json.loads(package.read_text(\'direct_url.json\') or \'{}\')\n            archive=direct.get(\'archive_info\',{})\n            report[\'packages\'][name]={\'version\':package.version,\'sha256\':archive.get(\'hashes\',{}).get(\'sha256\')}\n            if name==\'torch\':\n                path=Path(package.locate_file(\'torch/version.py\'))\n                # Read static CUDA build version without importing torch/initializing CUDA.\n                tree=ast.parse(path.read_text())\n                for item in tree.body:\n                    if isinstance(item,ast.AnnAssign) and isinstance(item.target,ast.Name) and item.target.id==\'cuda\':\n                        report[\'torch_cuda_build\']=ast.literal_eval(item.value)\n        except metadata.PackageNotFoundError:report[\'packages\'][name]=None\n        except Exception as error:report[\'packages\'][name]={\'error\':str(error)}\n    report[\'node_commit\']=command([\'git\',\'-C\',str(NODE),\'rev-parse\',\'HEAD\'])\n    report[\'node_config\']={name:text(NODE/name,5000) for name in [\n        \'trellis2_gguf/modules/sparse/conv/config.py\',\'trellis2_gguf/modules/sparse/config.py\']}\n    try:\n        with urllib.request.urlopen(\'http://127.0.0.1:8188/queue\',timeout=3) as response:queue=json.load(response)\n        report[\'queue\']={\'running\':len(queue.get(\'queue_running\',[])),\'pending\':len(queue.get(\'queue_pending\',[]))}\n        settings={}\n        allowed={\'device\',\'backend\',\'model_format\',\'modelname\',\'low_vram\',\'keep_models_loaded\',\'pipeline_type\',\n                 \'max_num_tokens\',\'sparse_structure_resolution\',\'use_tiled_decoder\',\'generate_texture_slat\',\'shape_steps\',\'sparse_structure_steps\'}\n        for item in queue.get(\'queue_running\',[]):\n            prompt=item[2]\n            for node_id,node in prompt.items():\n                if node.get(\'class_type\',\'\').startswith(\'Trellis\'):\n                    settings[node_id]={\'type\':node[\'class_type\'],\'settings\':{k:v for k,v in node.get(\'inputs\',{}).items() if k in allowed}}\n        report[\'running_settings\']=settings\n    except Exception as error:report[\'queue_error\']=str(error)\n    report[\'samples\']=[]\n    for index in range(samples):\n        row={\'index\':index,\'gpu\':command([\'nvidia-smi\',\'--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu,pstate\',\'--format=csv,noheader,nounits\']),\n             \'gpu_processes\':command([\'nvidia-smi\',\'--query-compute-apps=pid,used_gpu_memory\',\'--format=csv,noheader,nounits\'])}\n        if process_pid:\n            status=text(f\'/proc/{process_pid}/status\',10000) or \'\'\n            row[\'comfy_process\']=[line for line in status.splitlines() if line.startswith((\'State:\',\'VmRSS:\',\'VmPeak:\',\'Threads:\'))]\n        report[\'samples\'].append(row)\n        print(f\'GPU sample {index+1}/{samples}: \'+row[\'gpu\'].get(\'output\',str(row[\'gpu\'])).strip(),flush=True)\n        if index>=1 and all(\'error\' in r[\'gpu\'] for r in report[\'samples\'][-2:]):\n            report[\'sampling_note\']=\'Stopped after two consecutive nvidia-smi timeouts/errors.\'\n            break\n        if index+1<samples:time.sleep(1)\n    report[\'comfy_log_tail\']=text(ROOT/\'comfy-t4.log\',22000)\n    report[\'installed_manifest\']=text(ROOT/\'t4-installed.json\',5000)\n    return report\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--samples\',type=int,default=10)\n    args=parser.parse_args()\n    report=collect(max(1,min(args.samples,30)))\n    target=ROOT/\'trellis-diagnostics.json\'\n    target.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf8\')\n    print(\'Report saved:\',target,flush=True)\n',encoding='utf8')
    result=subprocess.run(['/content/venv312/bin/python','-u',str(script),'--samples','8'],capture_output=True,text=True,timeout=90)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
    files.download('/content/trellis-diagnostics.json')


## Необязательно: обновить только код ноды без сборок
Для уже установленной среды. Отметьте UPDATE_NODE_ONLY и выполните ячейку: она остановит сервер этого notebook, сохранит прежнюю ноду и загрузит указанную версию из вашей репы. Установленные пакеты, модели и wheels сохраняются. Затем выполните только ячейку запуска сервера.


In [ ]:
UPDATE_NODE_ONLY = False # @param {type:"boolean"}
if UPDATE_NODE_ONLY:
    import sys, importlib.util
    from pathlib import Path
    bootstrap=Path('/content/trellis-t4-bootstrap/colab')
    bootstrap.mkdir(parents=True,exist_ok=True)
    launcher=bootstrap/'server_t4.py'
    if launcher.exists():
        spec=importlib.util.spec_from_file_location('trellis_stop_for_update',launcher)
        manager=importlib.util.module_from_spec(spec)
        spec.loader.exec_module(manager)
        manager.stop()
    script=bootstrap/'update_node.py'
    script.write_text('"""Update only Python node code; retain CUDA wheels, models, and a full old checkout backup."""\nimport argparse\nimport ast\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport shutil\nimport socket\nimport subprocess\nimport time\nimport uuid\n\nROOT=Path(\'/content\')\nNODE=ROOT/\'ComfyUI/custom_nodes/ComfyUI-Trellis2-GGUF\'\nREPO=\'https://github.com/liblag/Trellis2-GGUF-ConfyUI.git\'\n\ndef update(ref):\n    with socket.socket() as sock:\n        if sock.connect_ex((\'127.0.0.1\',8188))==0:\n            raise RuntimeError(\'Stop the current ComfyUI process before updating node code. Do not reset the Colab runtime.\')\n    staging=ROOT/\'t4-node-update\'/uuid.uuid4().hex\n    staging.mkdir(parents=True)\n    def git(*args):subprocess.run([\'git\',\'-C\',str(staging),*args],check=True)\n    git(\'init\');git(\'remote\',\'add\',\'origin\',REPO);git(\'fetch\',\'--depth\',\'1\',\'origin\',ref);git(\'checkout\',\'--detach\',\'FETCH_HEAD\')\n    for name in [\'nodes.py\',\'__init__.py\',\'execution_logging.py\',\'gpu_residency.py\',\n                 \'trellis2_gguf/modules/sparse/conv/explicit_gemm.py\']:\n        path=staging/name\n        if not path.is_file():raise RuntimeError(\'Required updated source is missing: \'+name)\n        ast.parse(path.read_text(encoding=\'utf8\'),filename=name)\n    revision=subprocess.check_output([\'git\',\'-C\',str(staging),\'rev-parse\',\'HEAD\'],text=True).strip()\n    backup=ROOT/\'t4-node-backups\'/f\'{NODE.name}-{time.time_ns()}\'\n    backup.parent.mkdir(parents=True,exist_ok=True)\n    NODE.parent.mkdir(parents=True,exist_ok=True)\n    had_previous=NODE.exists()\n    if had_previous:shutil.move(str(NODE),str(backup))\n    try:shutil.move(str(staging),str(NODE))\n    except BaseException:\n        if had_previous and not NODE.exists():shutil.move(str(backup),str(NODE))\n        raise\n    for name in [\'trellis-t4-bootstrap/colab/sources.lock.json\',\'t4-installed.json\',\'t4-verified.json\']:\n        path=ROOT/name\n        if not path.exists():continue\n        data=json.loads(path.read_text())\n        sources=data.get(\'sources\',data)\n        if \'trellis\' in sources:\n            sources[\'trellis\']={\'url\':REPO,\'ref\':revision}\n            path.write_text(json.dumps(data,indent=2))\n    print(\'Updated node:\',revision)\n    if had_previous:print(\'Previous checkout preserved:\',backup)\n    print(\'CUDA wheels, venv312, input and output were not changed. Start ComfyUI again.\')\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--ref\',required=True)\n    update(parser.parse_args().ref)\n',encoding='utf8')
    helper=bootstrap/'process_output.py'
    helper.write_text('"""Stream subprocess output into notebook stdout and a persistent text log."""\nimport os\nfrom pathlib import Path\nimport queue\nimport signal\nimport subprocess\nimport threading\nimport time\n\n\ndef run_streamed(args, log_path, timeout=None, env=None):\n    args = list(map(str, args))\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    environment = dict(os.environ if env is None else env, PYTHONUNBUFFERED=\'1\')\n    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               text=True, encoding=\'utf8\', errors=\'replace\', bufsize=1,\n                               env=environment, start_new_session=os.name != \'nt\')\n    output = queue.Queue()\n    def read_output():\n        try:\n            for line in process.stdout:\n                output.put(line)\n        finally:\n            output.put(None)\n    reader = threading.Thread(target=read_output, daemon=True)\n    reader.start()\n    start = time.monotonic()\n    try:\n        with log_path.open(\'a\', encoding=\'utf8\', buffering=1) as log:\n            log.write(\'\\nCOMMAND: \'+\' \'.join(args)+\'\\n\')\n            ended = False\n            while not ended:\n                if timeout is not None and time.monotonic()-start > timeout:\n                    raise subprocess.TimeoutExpired(args, timeout)\n                try:\n                    line = output.get(timeout=.2)\n                except queue.Empty:\n                    continue\n                if line is None:\n                    ended = True\n                else:\n                    print(line, end=\'\', flush=True)\n                    log.write(line)\n            code = process.wait(timeout=5)\n            log.write(f\'EXIT CODE: {code}\\n\')\n        if code:\n            print(f\'Full subprocess log: {log_path}\', flush=True)\n            raise subprocess.CalledProcessError(code, args)\n        return subprocess.CompletedProcess(args, code)\n    finally:\n        if process.poll() is None:\n            if os.name == \'nt\': process.terminate()\n            else: os.killpg(process.pid, signal.SIGTERM)\n            try:\n                process.wait(timeout=3)\n            except subprocess.TimeoutExpired:\n                if os.name == \'nt\': process.kill()\n                else: os.killpg(process.pid, signal.SIGKILL)\n                process.wait()\n        reader.join(timeout=2)\n        process.stdout.close()\n',encoding='utf8')
    spec=importlib.util.spec_from_file_location('trellis_update_output',helper)
    output=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(output)
    result=output.run_streamed([sys.executable,'-u',str(script),'--ref','bee72a2bca87b6e9b1136dcf99b71318d456a913'],'/content/colab-update-node.log')
    print('Код ноды обновлён. Теперь выполните ячейку «Запуск / перезапуск ComfyUI». Установку запускать не нужно.')
